In [ ]:
import streamlit as st
import pandas as pd
import joblib
import geopandas as gpd
import plotly.express as px
import pickle
import pandas as pd
import numpy as np

# load the pickled object safely from file
with open("best_features_arr.pkl", "rb") as _f:
    arr_cats = pickle.load(_f)
with open('best_features_ven.pkl', 'rb') as _f:
    ven_cats = pickle.load(_f)
with open('xgb_model_arr_med.pkl', 'rb') as _f:
    xgb_model_arr_med = pickle.load(_f)
with open('xgb_model_ven_med.pkl', 'rb') as _f:
    xgb_model_ven_med = pickle.load(_f)
with open('list_barrios.pkl', 'rb') as _f:
    list_barrios = pickle.load(_f)
with open('price_per_m2_ven.pkl', 'rb') as _f:
    ppmc_ven = pickle.load(_f)
with open('price_per_space_ven.pkl', 'rb') as _f:
    pppz_ven = pickle.load(_f)
with open('price_per_m2_arr.pkl', 'rb') as _f:
    ppmc_arr = pickle.load(_f)
with open('price_per_space_arr.pkl', 'rb') as _f:
    pppz_arr = pickle.load(_f)
with open('preprocessor.pkl', 'rb') as _f:
    preprocessor = pickle.load(_f)
with open('price_per_parking_arr.pkl', 'rb') as _f:
    pppp_arr = pickle.load(_f)
with open('price_per_parking_ven.pkl', 'rb') as _f:
    pppp_ven = pickle.load(_f)
with open('cat_model_arr_med.pkl', 'rb') as _f:
    cat_model_arr_med = pickle.load(_f)
loan = pd.read_csv('arr_mede_final.csv')
sales = pd.read_csv('ven_mede_final.csv')
gdf = gpd.read_file("medellin.geojson")

def pred_func_arr(area, habitaciones, banos, parqueaderos, barrio, tipo):
    """
    Make prediction for rental price using the trained pipeline
    """
    # Normalizar y comprobar barrio (case-insensitive)
    barrio_norm = str(barrio).strip().lower()
    barrios_lc = [b.strip().lower() for b in list_barrios]
    if barrio_norm not in barrios_lc:
        return f'El barrio \"{barrio}\" no está en la lista de barrios conocidos. Por favor, elija uno de los siguientes: {list_barrios}'
    tipo_norm = str(tipo).strip().lower()
    tipos_lc = [t.strip().lower() for t in loan['tipo'].unique().tolist()]
    if tipo_norm not in tipos_lc:
        return f'El tipo \"{tipo}\" no está en la lista de tipos conocidos. Por favor, elija uno de los siguientes: {loan["tipo"].unique().tolist()}'
    # Crear DataFrame de entrada con las columnas esperadas por el preprocessor
    cols = arr_cats
    input_df = pd.DataFrame([{
        'habitaciones': habitaciones,
        'baños': banos,
        'parqueaderos': parqueaderos,
        'espacios': None,  # se calculará abajo
        'axe': None,
        'tipo': tipo,
        'ppmc': None,
        'pppz': None,
        'garaje_bin': None,
        'parq2': None,
        'new_index': None,
        'area': area,
        'barrio': barrio,
        'axh': None,
        'axa': None
    }])
    
    # Calcular espacios y axe (asegurando no división por cero)
    input_df['espacios'] = input_df['habitaciones'] + input_df['parqueaderos'] + input_df['baños']
    input_df['axe'] = input_df['area'] / input_df['espacios'].replace({0: pd.NA})
    input_df['axh'] = input_df['area'] / input_df['habitaciones'].replace({0: pd.NA})
    input_df['axa'] = input_df['area'] * input_df['area']
    input_df['parq2'] = input_df['parqueaderos'] * input_df['parqueaderos']
    input_df['ppmc'] = input_df['barrio'].map(ppmc_arr)
    input_df['pppz'] = input_df['barrio'].map(pppz_arr)
    input_df['new_index'] = input_df['ppmc']/(input_df['ppmc'].max())*100
    input_df['garaje_bin'] = input_df['parqueaderos'].apply(lambda x: 1 if x > 0 else 0)
    input_df['pppp'] = input_df['barrio'].map(pppp_arr)
    input_df['pppp/pppz'] = (input_df['pppp'] / input_df['pppz']).replace([np.inf, -np.inf], 0).fillna(1).astype(float)
    input_df['pppp/ppmc'] = (input_df['pppp'] / input_df['ppmc']).replace([np.inf, -np.inf], 0).fillna(1).astype(float)
    input_df['pppz/ppmc'] = (input_df['pppz'] / input_df['ppmc']).replace([np.inf, -np.inf], 0).fillna(1).astype(float)
    
    # Transformar datos de entrada con el preprocessor ya entrenado
    input_transformed = preprocessor.transform(input_df)

    # Si es matriz dispersa, convertir a densa
    if hasattr(input_transformed, "toarray"):
        arr = input_transformed.toarray()
    else:
        arr = input_transformed

    # Obtener nombres de características generadas por el preprocessor (fallback si no está disponible)
    try:
        feature_names = preprocessor.get_feature_names_out()
    except Exception:
        feature_names = [f"f_{i}" for i in range(arr.shape[1])]

    # Convertir a DataFrame para poder indexar por nombre de columna
    input_transformed_df = pd.DataFrame(arr, columns=feature_names)

    # Comprobar que las columnas esperadas están presentes
    missing_cols = [c for c in cols if c not in input_transformed_df.columns]
    if missing_cols:
        raise KeyError(f"Faltan columnas después de la transformación: {missing_cols}")

    pred = xgb_model_arr_med.predict(input_transformed_df[cols])
    return np.expm1(pred) 

def pred_func_ven(area, habitaciones, banos, parqueaderos, barrio, tipo):
    """
    Make prediction for rental price using the trained pipeline
    """
    # Normalizar y comprobar barrio (case-insensitive)
    barrio_norm = str(barrio).strip().lower()
    barrios_lc = [b.strip().lower() for b in list_barrios]
    if barrio_norm not in barrios_lc:
        return f'El barrio \"{barrio}\" no está en la lista de barrios conocidos. Por favor, elija uno de los siguientes: {list_barrios}'
    tipo_norm = str(tipo).strip().lower()
    tipos_lc = [t.strip().lower() for t in sales['tipo'].unique().tolist()]
    if tipo_norm not in tipos_lc:
        return f'El tipo \"{tipo}\" no está en la lista de tipos conocidos. Por favor, elija uno de los siguientes: {sales["tipo"].unique().tolist()}'
    # Crear DataFrame de entrada con las columnas esperadas por el preprocessor
    cols = ven_cats
    input_df = pd.DataFrame([{
        'habitaciones': habitaciones,
        'baños': banos,
        'parqueaderos': parqueaderos,
        'espacios': None,  # se calculará abajo
        'axe': None,
        'tipo': tipo,
        'ppmc': None,
        'pppz': None,
        'garaje_bin': None,
        'parq2': None,
        'new_index': None,
        'area': area,
        'barrio': barrio,
        'axh': None,
        'axa': None
    }])
    
    # Calcular espacios y axe (asegurando no división por cero)
    input_df['espacios'] = input_df['habitaciones'] + input_df['parqueaderos'] + input_df['baños']
    input_df['axe'] = input_df['area'] / input_df['espacios'].replace({0: pd.NA})
    input_df['axh'] = input_df['area'] / input_df['habitaciones'].replace({0: pd.NA})
    input_df['axa'] = input_df['area'] * input_df['area']
    input_df['parq2'] = input_df['parqueaderos'] * input_df['parqueaderos']
    input_df['ppmc'] = input_df['barrio'].map(ppmc_ven)
    input_df['pppz'] = input_df['barrio'].map(pppz_ven)
    input_df['new_index'] = input_df['ppmc']/(input_df['ppmc'].max())*100
    input_df['garaje_bin'] = input_df['parqueaderos'].apply(lambda x: 1 if x > 0 else 0)
    input_df['pppp'] = input_df['barrio'].map(pppp_ven)
    input_df['pppp/pppz'] = (input_df['pppp'] / input_df['pppz']).replace([np.inf, -np.inf], 0).fillna(1).astype(float)
    input_df['pppp/ppmc'] = (input_df['pppp'] / input_df['ppmc']).replace([np.inf, -np.inf], 0).fillna(1).astype(float)
    input_df['pppz/ppmc'] = (input_df['pppz'] / input_df['ppmc']).replace([np.inf, -np.inf], 0).fillna(1).astype(float)
    
    # Transformar datos de entrada con el preprocessor ya entrenado
    input_transformed = preprocessor.transform(input_df)

    # Si es matriz dispersa, convertir a densa
    if hasattr(input_transformed, "toarray"):
        arr = input_transformed.toarray()
    else:
        arr = input_transformed

    # Obtener nombres de características generadas por el preprocessor (fallback si no está disponible)
    try:
        feature_names = preprocessor.get_feature_names_out()
    except Exception:
        feature_names = [f"f_{i}" for i in range(arr.shape[1])]

    # Convertir a DataFrame para poder indexar por nombre de columna
    input_transformed_df = pd.DataFrame(arr, columns=feature_names)

    # Comprobar que las columnas esperadas están presentes
    missing_cols = [c for c in cols if c not in input_transformed_df.columns]
    if missing_cols:
        raise KeyError(f"Faltan columnas después de la transformación: {missing_cols}")

    pred = xgb_model_ven_med.predict(input_transformed_df[cols])
    return np.expm1(pred) 

  


In [3]:
preprocessor

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('scaler', StandardScaler())]),
                                 ['area', 'baños', 'parqueaderos', 'espacios',
                                  'pppz', 'garaje_bin', 'ppmc', 'axe',
                                  'habitaciones', 'axh', 'axa', 'new_index',
                                  'parq2', 'pppp', 'pppp/pppz', 'pppp/ppmc',
                                  'pppz/ppmc']),
                                ('cat',
                                 Pipeline(steps=[('ohe',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['tipo'])])

In [6]:
pred_func_ven(100, 3, 2, 1, 'rosales', 'Apartamento')

array([6.5151763e+08], dtype=float32)

In [33]:
pred_func_arr(180, 4, 2, 0, 'rosales', 'Apartamento')

array([2.7335828e+06], dtype=float32)

In [ ]:
#Creacion de categorias de oportunidades de compra y arriendo
loan_standarized = preprocessor.transform(loan)
loan_standarized = pd.DataFrame(loan_standarized, columns=preprocessor.get_feature_names_out())
loan['cat_pred'] = np.exp(cat_model_arr_med.predict(loan_standarized[arr_cats]))
loan['is underpriced_cat'] = loan['precio'] < loan['cat_pred']
loan['how much underpriced_cat'] = (loan['cat_pred'] - loan['precio'])/loan['cat_pred']*100
loan['oportunity_houses'] = loan['how much underpriced_cat'] > 20

sales_standarized = preprocessor.transform(sales)
sales_standarized = pd.DataFrame(sales_standarized, columns=preprocessor.get_feature_names_out())
sales['cat_pred'] = np.exp(xgb_model_ven_med.predict(sales_standarized[ven_cats]))
sales['is underpriced_cat'] = sales['precio'] < sales['cat_pred']
sales['how much underpriced_cat'] = (sales['cat_pred'] - sales['precio'])/sales['cat_pred']*100
sales['oportunity_houses'] = sales['how much underpriced_cat'] > 20